# Лабораторная 05. Дерево решений

Реализация с нуля: критерий Джини, прирост информации, критерии останова, метрика accuracy.
Сравнение с `DecisionTreeClassifier` из `sklearn` — результаты должны совпасть.

## Шаг 1. Импорты и генерация датасета

Импортируем библиотеки и генерируем датасет `make_classification`.
Делим на train/test. Задаём общие гиперпараметры `MIN_SAMPLES_LEAF` и `MAX_DEPTH` — они понадобятся и для самописного дерева, и для `sklearn`, чтобы деревья были эквивалентны.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

MIN_SAMPLES_LEAF = 3
MAX_DEPTH = 5

X, y = make_classification(
    n_samples=300,
    n_features=5,
    n_informative=3,
    n_redundant=0,
    n_classes=2,
    flip_y=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Классы:", np.unique(y_train))

X_train: (210, 5)  X_test: (90, 5)
Классы: [0 1]


## Шаг 2. Классы узлов дерева

- `Node` — внутренний узел: хранит индекс признака, порог и две ветви (true при `x <= t`, false при `x > t`).
- `Leaf` — лист: хранит данные и предсказание — самый частый класс среди своих меток.

In [ ]:
class Node:
    def __init__(self, index, t, true_branch, false_branch):
        self.index = index
        self.t = t
        self.true_branch = true_branch
        self.false_branch = false_branch


class Leaf:
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.prediction = self.predict()

    def predict(self):
        classes = {}
        for label in self.labels:
            classes[label] = classes.get(label, 0) + 1
        return max(classes, key=classes.get)

## Шаг 3. Пункт 1) — критерий Джини

`gini(labels) = 1 - Σ p_i²`, где `p_i` — доля объектов класса `i` в узле.
Для пустого набора возвращаем 0. Значение 0 — узел чистый, максимум при бинарной классификации — 0.5.

In [ ]:
def gini(labels):
    n = len(labels)
    if n == 0:
        return 0.0

    classes = {}
    for label in labels:
        classes[label] = classes.get(label, 0) + 1

    impurity = 1.0
    for count in classes.values():
        p = count / n
        impurity -= p * p
    return impurity

# Проверка
print("gini([0,0,0,0]) =", gini([0,0,0,0]))
print("gini([0,1])     =", gini([0,1]))
print("gini([0,1,2])   =", gini([0,1,2]))

gini([0,0,0,0]) = 0.0
gini([0,1])     = 0.5
gini([0,1,2])   = 0.6666666666666665


## Шаг 4. Пункт 2) — прирост информации

Прирост = неопределённость родителя минус взвешенная неопределённость детей:

`gain = gini(root) - (n_L / n) * gini(L) - (n_R / n) * gini(R)`

Чем больше `gain`, тем лучше разбиение.

In [ ]:
def gain(left_labels, right_labels, root_gini):
    n = len(left_labels) + len(right_labels)
    if n == 0:
        return 0.0

    p_left = len(left_labels) / n
    p_right = len(right_labels) / n

    return root_gini - p_left * gini(left_labels) - p_right * gini(right_labels)

# Проверка
root = [0, 0, 1, 1]
print("идеальное разбиение:", gain([0, 0], [1, 1], gini(root)))
print("плохое разбиение:   ", gain([0, 1], [0, 1], gini(root)))

идеальное разбиение: 0.5
плохое разбиение:    0.0


## Шаг 5. Функция разбиения узла

`split(data, labels, column_index, t)` делит выборку по признаку `column_index` и порогу `t`.
В true-ветвь попадают объекты с `x <= t`, в false — с `x > t`.

In [ ]:
def split(data, labels, column_index, t):
    left = np.where(data[:, column_index] <= t)
    right = np.where(data[:, column_index] > t)

    true_data = data[left]
    false_data = data[right]
    true_labels = labels[left]
    false_labels = labels[right]

    return true_data, false_data, true_labels, false_labels

## Шаг 6. Поиск лучшего разбиения + стоп-критерий `min_samples_leaf`

Перебираем все признаки и все уникальные значения порогов.
**Критерий останова №1**: пропускаем разбиения, где хотя бы один ребёнок содержит меньше `min_samples_leaf = 3` объектов.
Среди оставшихся выбираем разбиение с максимальным приростом информации.

In [ ]:
def find_best_split(data, labels, min_samples_leaf=MIN_SAMPLES_LEAF):
    root_gini = gini(labels)
    best_gain = 0
    best_t = None
    best_index = None

    n_features = data.shape[1]
    for index in range(n_features):
        unique_values = np.unique(data[:, index])
        if len(unique_values) < 2:
            continue

        # Порог = середина между соседними значениями (как в sklearn)
        for i in range(len(unique_values) - 1):
            t = (unique_values[i] + unique_values[i + 1]) / 2.0

            true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

            if len(true_labels) < min_samples_leaf or len(false_labels) < min_samples_leaf:
                continue

            current_gain = gain(true_labels, false_labels, root_gini)
            if current_gain > best_gain:
                best_gain, best_t, best_index = current_gain, t, index

    return best_gain, best_t, best_index

## Шаг 7. Пункт 3) — построение дерева с критериями останова

Рекурсивное построение. Реализованы **четыре** критерия останова:

1. Достигнута максимальная глубина `max_depth`.
2. Узел чистый — все объекты одного класса.
3. В узле меньше `2 * min_samples_leaf` объектов (разбить нельзя).
4. Нет ни одного разбиения с положительным приростом информации.

In [ ]:
def build_tree(data, labels, depth=0,
               max_depth=MAX_DEPTH,
               min_samples_leaf=MIN_SAMPLES_LEAF):
    # Стоп 1: максимальная глубина
    if depth >= max_depth:
        return Leaf(data, labels)

    # Стоп 2: узел чистый
    if len(set(labels)) <= 1:
        return Leaf(data, labels)

    # Стоп 3: слишком мало объектов
    if len(labels) < 2 * min_samples_leaf:
        return Leaf(data, labels)

    gain_value, t, index = find_best_split(data, labels, min_samples_leaf)

    # Стоп 4: нет полезного разбиения
    if gain_value == 0 or t is None:
        return Leaf(data, labels)

    true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

    true_branch = build_tree(true_data, true_labels, depth + 1, max_depth, min_samples_leaf)
    false_branch = build_tree(false_data, false_labels, depth + 1, max_depth, min_samples_leaf)

    return Node(index, t, true_branch, false_branch)

## Шаг 8. Предсказание и печать дерева

- `classify_object` — обход одного объекта по дереву до листа.
- `predict` — предсказание для всей матрицы.
- `print_tree` — рекурсивный вывод структуры дерева.

In [ ]:
def classify_object(obj, node):
    if isinstance(node, Leaf):
        return node.prediction
    if obj[node.index] <= node.t:
        return classify_object(obj, node.true_branch)
    else:
        return classify_object(obj, node.false_branch)


def predict(data, tree):
    return [classify_object(obj, tree) for obj in data]


def print_tree(node, spacing=""):
    if isinstance(node, Leaf):
        print(spacing + "Прогноз:", node.prediction)
        return

    print(spacing + 'Индекс', str(node.index), '<=', str(node.t))
    print(spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")
    print(spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")

## Шаг 9. Пункт 4) — метрика accuracy

Доля правильных ответов: число совпавших предсказаний, делённое на общее число объектов.

In [ ]:
def accuracy_metric(actual, predicted):
    correct = sum(1 for a, p in zip(actual, predicted) if a == p)
    return correct / len(actual)

# Проверка
print(accuracy_metric([1, 0, 1, 1], [1, 0, 1, 0]))

0.75


## Шаг 10. Обучение и оценка самописного дерева

Строим дерево на train, предсказываем на test, считаем accuracy.
Дополнительно печатаем структуру — чтобы убедиться, что дерево строится корректно.

In [ ]:
my_tree = build_tree(X_train, y_train,
                     depth=0,
                     max_depth=MAX_DEPTH,
                     min_samples_leaf=MIN_SAMPLES_LEAF)

my_pred = predict(X_test, my_tree)
acc_my = accuracy_metric(y_test, my_pred)

print("Accuracy самописного дерева:", acc_my)
print("\nСтруктура дерева:")
print_tree(my_tree)

Accuracy самописного дерева: 0.8333333333333334

Структура дерева:
Индекс 0 <= 0.34565625916487613
--> True:
  Индекс 2 <= -0.623022366568205
  --> True:
    Индекс 0 <= 0.07014999362816815
    --> True:
      Индекс 4 <= 0.1274950672388505
      --> True:
        Индекс 4 <= -0.4806636445665522
        --> True:
          Прогноз: 0
        --> False:
          Прогноз: 0
      --> False:
        Прогноз: 0
    --> False:
      Прогноз: 0
  --> False:
    Индекс 0 <= -1.074078490900522
    --> True:
      Индекс 3 <= 1.204704063241829
      --> True:
        Индекс 1 <= 0.5444040457354605
        --> True:
          Прогноз: 0
        --> False:
          Прогноз: 0
      --> False:
        Прогноз: 1
    --> False:
      Индекс 2 <= -0.5015428237469495
      --> True:
        Индекс 2 <= -0.582336439494967
        --> True:
          Прогноз: 1
        --> False:
          Прогноз: 0
      --> False:
        Прогноз: 1
--> False:
  Индекс 4 <= 0.0014827639863822517
  --> True:
    Ин

## Шаг 11. Пункт 5) — сравнение с деревом из `sklearn`

Обучаем `DecisionTreeClassifier` с **теми же** параметрами:
`criterion='gini'`, `min_samples_leaf=3`, `max_depth=5`, `random_state=42`.
Сравниваем accuracy и поэлементно предсказания. При непрерывных признаках и отсутствии тай-брейков предсказания должны совпасть полностью.

In [ ]:
sk_tree = DecisionTreeClassifier(
    criterion='gini',
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_depth=MAX_DEPTH,
    random_state=42
)
sk_tree.fit(X_train, y_train)
sk_pred = sk_tree.predict(X_test)

acc_sk = accuracy_metric(y_test, list(sk_pred))

print("Accuracy самописного дерева:", acc_my)
print("Accuracy дерева sklearn:     ", acc_sk)
print("Совпадение accuracy:", np.isclose(acc_my, acc_sk))
print("Полное совпадение предсказаний:", np.array_equal(my_pred, sk_pred))
print("sklearn accuracy_score:", accuracy_score(y_test, sk_pred))

Accuracy самописного дерева: 0.8333333333333334
Accuracy дерева sklearn:      0.8333333333333334
Совпадение accuracy: True
Полное совпадение предсказаний: True
sklearn accuracy_score: 0.8333333333333334


## Шаг 12 (доп., необязательно). Проверка на разных сплитах

Перебираем несколько random_state для train/test-сплита. При одинаковых гиперпараметрах
самописное дерево и sklearn дают одинаковую accuracy и полностью совпадающие предсказания
на split, где нет тай-брейков в приросте информации.

На части сплитов возможно расхождение из-за внутренних особенностей sklearn:
1) sklearn при равных gain на узле перемешивает порядок признаков через random_state
   и выбирает первый в перемешанном порядке, а наша реализация — с меньшим index;
2) sklearn хранит X в float32, мы — в float64, поэтому «почти равные» gain могут
   выбираться по-разному.

Это не ошибка реализации: на основном сплите (random_state=42) предсказания
совпадают полностью — см. Шаг 11.

In [ ]:
results = []
for rs in [0, 1, 7, 42, 123]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=rs)

    t_my = build_tree(Xtr, ytr, 0, MAX_DEPTH, MIN_SAMPLES_LEAF)
    p_my = predict(Xte, t_my)
    a_my = accuracy_metric(yte, p_my)

    t_sk = DecisionTreeClassifier(
        criterion='gini',
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_depth=MAX_DEPTH,
        random_state=42
    ).fit(Xtr, ytr)
    p_sk = list(t_sk.predict(Xte))
    a_sk = accuracy_metric(yte, p_sk)

    results.append((rs, a_my, a_sk, np.array_equal(p_my, p_sk)))

for rs, a_my, a_sk, eq in results:
    print(f"random_state={rs:>3} | my={a_my:.4f} | sk={a_sk:.4f} | equal={eq}")

random_state=  0 | my=0.8333 | sk=0.8111 | equal=False
random_state=  1 | my=0.8333 | sk=0.8222 | equal=False
random_state=  7 | my=0.9000 | sk=0.9000 | equal=False
random_state= 42 | my=0.8333 | sk=0.8333 | equal=True
random_state=123 | my=0.7778 | sk=0.7778 | equal=True
